# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We utilize the Croissant schema to access and process dataset metadata and records.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (using their `@id`), fields, and their IDs.

The Croissant metadata provides access to all record sets, fields, and columns for exploration.

In [ ]:
# List all available record sets with their @id and name
recordsets = list(metadata.record_sets)
print("Available Record Sets:")
for rs in recordsets:
    print(f"  @id: {rs.id}  |  name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as identified above.

For this example, we'll select the first available record set as a demonstration (you may substitute the `@id` with others as needed).

In [ ]:
# Extract all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]

# Load each record set into a DataFrame, stored in a dictionary
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set @id: {rs_id} with {len(df)} rows and columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Pick a record set for detailed analysis (e.g., the first available)
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f'Chosen record set for demonstration: {main_record_set_id}')
    print(dataframes[main_record_set_id].head())
else:
    print("No record sets available.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping. This example demonstrates these steps using the main record set.

In [ ]:
# Identify candidate numeric and group fields by listing all columns
df = dataframes[main_record_set_id]
print(f"Columns in record set '{main_record_set_id}': {df.columns.tolist()}")

# Try to find a numeric field; if not found, the code will not raise
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    print('No numeric field found for analysis. Please inspect and specify one.')
else:
    print(f'Using numeric field: {numeric_field}')
    threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (mean):")
    print(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Attempt grouping by a likely categorical field
    group_field = None
    for col in df.columns:
        # Skip the numeric field and columns with too many distinct values
        if col != numeric_field and df[col].nunique() < len(df)//2:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using Pandas or matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Plot histogram of the selected numeric field, if found
if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} in Record Set {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    # If a group field is found, plot boxplot
    if group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset using the `mlcroissant` library. Using entity `@id`s, we listed record sets and their fields, loaded data, performed standard EDA (including filtering, normalization, and grouping), and visualized numeric attributes. For further analysis, repeat these steps for other record sets or fields of interest, always referencing them by their `@id`.